# Step 2 — Correlation Analysis Between Series

This notebook continues directly from **Step 1**.  
It expects the following DataFrames to be already available (produced by Step 1):

| DataFrame | Frequency | Columns |
|---|---|---|
| `weekly_all` | Weekly (W) | `gtrends`, `neg_art_count`, `avg_tone` |
| `daily_all` | Daily (D) | `gtrends`, `neg_art_count`, `avg_tone` |

**Pairs analysed:**
1. Google Trends vs Negative article counts  
2. Google Trends vs Average tone  
3. Negative article counts vs Average tone ← multicollinearity check for β(t)

**Both in levels and in first differences** (Pearson + Spearman).

In [ ]:
# ── Re-run Step 1 to produce weekly_all and daily_all ────────────────────────
# (paste or %run your Step 1 notebook here if running standalone)

import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import adfuller

# --- LOAD GDELT DAILY ---
neg_counts = pd.read_csv('Articles_count_Negative_keywords_daily.csv', sep=';')
tone       = pd.read_csv('Average_tone_Neutral_keywords_daily.csv', sep=';')
neu_counts = pd.read_csv('Articles_count_daily_Neutral_keywords_daily.csv', sep=';')

for df in [neg_counts, tone, neu_counts]:
    date_col = df.columns[0]
    df['date'] = pd.to_datetime(df[date_col], dayfirst=True)
    df.set_index('date', inplace=True)
    if date_col != 'date':
        df.drop(columns=[date_col], inplace=True)

neg_val  = neg_counts.columns[0]
tone_val = tone.columns[0]
neu_val  = neu_counts.columns[0]

# --- WEEKLY AGGREGATION ---
weekly_neg_counts = neg_counts[neg_val].resample('W').sum()
combined = pd.DataFrame({'tone': tone[tone_val], 'weight': neu_counts[neu_val]}).dropna()
combined['weighted_tone'] = combined['tone'] * combined['weight']
weekly_agg = combined.resample('W').sum()
weekly_avg_tone = weekly_agg['weighted_tone'] / weekly_agg['weight']

# Daily series
daily_neg  = neg_counts[neg_val]
daily_tone = tone[tone_val]

# Convert GTrends indices to Timestamp
naive_denton.index = pd.to_datetime(naive_denton.index)
weekly_final.index = pd.to_datetime(weekly_final.index)

# --- REINDEX DAILY ---
full_daily_range = pd.date_range(
    min(naive_denton.index.min(), daily_neg.index.min(), daily_tone.index.min()),
    max(naive_denton.index.max(), daily_neg.index.max(), daily_tone.index.max()),
    freq='D'
)
daily_all = pd.DataFrame({
    'gtrends':       naive_denton.reindex(full_daily_range),
    'neg_art_count': daily_neg.reindex(full_daily_range),
    'avg_tone':      daily_tone.reindex(full_daily_range)
}, index=full_daily_range)
daily_all.index.name = 'date'

# --- REINDEX WEEKLY ---
full_weekly_range = pd.date_range(
    min(weekly_final.index.min(), weekly_neg_counts.index.min(), weekly_avg_tone.index.min()),
    max(weekly_final.index.max(), weekly_neg_counts.index.max(), weekly_avg_tone.index.max()),
    freq='W'
)
weekly_all = pd.DataFrame({
    'gtrends':       weekly_final.reindex(full_weekly_range),
    'neg_art_count': weekly_neg_counts.reindex(full_weekly_range),
    'avg_tone':      weekly_avg_tone.reindex(full_weekly_range)
}, index=full_weekly_range)
weekly_all.index.name = 'week_end'

print(f'weekly_all shape: {weekly_all.shape}')
print(f'daily_all  shape: {daily_all.shape}')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

NICE = {
    'gtrends':       'Google Trends',
    'neg_art_count': 'Neg. article count',
    'avg_tone':      'Avg tone (weighted)'
}

# Work with the clean (dropna) weekly series
W = weekly_all.dropna().copy()
print(f'Clean weekly obs: {len(W)}  ({W.index[0].date()} → {W.index[-1].date()})')
W.describe().round(3)

## 1. Visual inspection of the three weekly series

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 7), sharex=True)
colors = ['#2196F3', '#FF5722', '#4CAF50']

for ax, col, color in zip(axes, W.columns, colors):
    ax.plot(W.index, W[col], color=color, linewidth=1.2)
    ax.set_ylabel(NICE[col], fontsize=10)
    ax.grid(True, alpha=0.3)

axes[0].set_title('Clean weekly series (Step 1 output)', fontsize=13, fontweight='bold')
axes[-1].set_xlabel('Week end date')
plt.tight_layout()
plt.show()

## 2. Correlations in levels

Pearson (linear) and Spearman (rank, robust to outliers/non-normality) for all three pairs.

In [ ]:
PAIRS = [
    ('gtrends',       'neg_art_count', 'GT vs Neg.articles'),
    ('gtrends',       'avg_tone',      'GT vs Tone'),
    ('neg_art_count', 'avg_tone',      'Neg.articles vs Tone'),
]

def corr_table(data, pairs, label=''):
    rows = []
    for xc, yc, name in pairs:
        x, y = data[xc].dropna(), data[yc].dropna()
        idx  = x.index.intersection(y.index)
        x, y = x[idx], y[idx]
        pr, pp = stats.pearsonr(x, y)
        sr, sp = stats.spearmanr(x, y)
        rows.append({'Pair': name,
                     'n': len(x),
                     'Pearson r': round(pr, 4), 'Pearson p': round(pp, 4),
                     'Spearman ρ': round(sr, 4), 'Spearman p': round(sp, 4)})
    tbl = pd.DataFrame(rows).set_index('Pair')
    print(f'=== {label} ===')
    return tbl

corr_lev = corr_table(W, PAIRS, 'Correlations IN LEVELS')
corr_lev

## 3. Correlations in first differences

Δyₜ = yₜ − yₜ₋₁ strips out shared long-run trends → reveals genuine week-to-week co-movement.

In [ ]:
W_diff = W.diff().dropna()

PAIRS_D = [
    ('gtrends',       'neg_art_count', 'ΔGT vs ΔNeg.articles'),
    ('gtrends',       'avg_tone',      'ΔGT vs ΔTone'),
    ('neg_art_count', 'avg_tone',      'ΔNeg.articles vs ΔTone'),
]

corr_dif = corr_table(W_diff, PAIRS_D, 'Correlations IN FIRST DIFFERENCES')
corr_dif

### Heatmap comparison: levels vs first differences

In [ ]:
Wr  = W.rename(columns=NICE)
Wdr = W_diff.rename(columns=NICE)

fig, axes = plt.subplots(2, 2, figsize=(11, 9))
configs = [
    (Wr,  'pearson',  'Pearson — levels'),
    (Wr,  'spearman', 'Spearman — levels'),
    (Wdr, 'pearson',  'Pearson — first differences'),
    (Wdr, 'spearman', 'Spearman — first differences'),
]

for ax, (data, method, title) in zip(axes.flat, configs):
    mat = data.corr(method=method)
    sns.heatmap(mat, ax=ax, annot=True, fmt='.3f', cmap='RdBu_r',
                vmin=-1, vmax=1, linewidths=0.5, annot_kws={'size': 12})
    ax.set_title(title, fontweight='bold')
    ax.tick_params(axis='x', rotation=20)
    ax.tick_params(axis='y', rotation=0)

plt.suptitle('Correlation matrices — levels vs first differences', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### Scatter plots — levels

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (xc, yc, _) in zip(axes, PAIRS):
    x, y = W[xc], W[yc]
    ax.scatter(x, y, alpha=0.4, s=15, color='steelblue')
    m, b = np.polyfit(x, y, 1)
    xs = np.linspace(x.min(), x.max(), 100)
    ax.plot(xs, m*xs + b, color='crimson', lw=1.5)
    pr, _ = stats.pearsonr(x, y)
    sr, _ = stats.spearmanr(x, y)
    ax.set_xlabel(NICE[xc]); ax.set_ylabel(NICE[yc])
    ax.set_title(f'r = {pr:.3f}   ρ = {sr:.3f}', fontsize=10)
    ax.grid(True, alpha=0.3)

plt.suptitle('Scatter plots — levels', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### Scatter plots — first differences

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (xc, yc, _) in zip(axes, PAIRS_D):
    x, y = W_diff[xc], W_diff[yc]
    ax.scatter(x, y, alpha=0.4, s=15, color='darkorange')
    m, b = np.polyfit(x, y, 1)
    xs = np.linspace(x.min(), x.max(), 100)
    ax.plot(xs, m*xs + b, color='navy', lw=1.5)
    pr, _ = stats.pearsonr(x, y)
    sr, _ = stats.spearmanr(x, y)
    ax.set_xlabel(NICE[xc.lstrip('Δ')]); ax.set_ylabel(NICE[yc.lstrip('Δ')])
    ax.set_title(f'Δ: r = {pr:.3f}   ρ = {sr:.3f}', fontsize=10)
    ax.axhline(0, color='grey', lw=0.7, ls='--')
    ax.axvline(0, color='grey', lw=0.7, ls='--')
    ax.grid(True, alpha=0.3)

plt.suptitle('Scatter plots — first differences', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Multicollinearity check: Neg. article counts vs Avg tone

Including both as regressors in β(t) is problematic if |r| or |ρ| > **0.70** (rule of thumb).  
If highly correlated → use only one, or orthogonalise (residualise one on the other).

In [ ]:
THRESHOLD = 0.70

def multicol_check(data, xc, yc, label):
    x, y = data[xc].dropna(), data[yc].dropna()
    idx  = x.index.intersection(y.index)
    pr, _ = stats.pearsonr(x[idx], y[idx])
    sr, _ = stats.spearmanr(x[idx], y[idx])
    flag = '⚠️  HIGH' if max(abs(pr), abs(sr)) > THRESHOLD else '✓  OK'
    print(f'{label:30s}  Pearson r={pr:+.4f}  Spearman ρ={sr:+.4f}  → {flag}')

print('─── Multicollinearity: Neg.article counts vs Avg tone ───')
multicol_check(W,      'neg_art_count', 'avg_tone', 'Levels')
multicol_check(W_diff, 'neg_art_count', 'avg_tone', 'First differences')

print()
pr_lev, _ = stats.pearsonr(*[W[c].values for c in ['neg_art_count','avg_tone']])
if abs(pr_lev) > THRESHOLD:
    print('  → Consider using only one regressor or residualising one on the other.')
else:
    print('  → Both regressors can tentatively be included; confirm with VIF before final model.')

## 5. Full correlation summary

In [ ]:
lev_labeled = corr_lev.copy()
lev_labeled.index = [f'{i} (levels)' for i in lev_labeled.index]
dif_labeled = corr_dif.copy()
dif_labeled.index = [f'{i} (Δ)' for i in dif_labeled.index]

full = pd.concat([lev_labeled, dif_labeled])
print('=== Full correlation summary ===')
full

## 6. Interpretation guide

| Finding | Implication |
|---|---|
| r(levels) >> r(differences) | Level correlation is inflated by shared trend; trust differences |
| r(GT, articles) > 0 in Δ | Public search interest and media coverage move together week-to-week |
| r(GT, tone) < 0 in Δ | More negative tone accompanies spikes in search interest (panic signal) |
| \|r(articles, tone)\| > 0.70 | Multicollinearity — do not include both in β(t) without correction |
| \|r(articles, tone)\| ≤ 0.70 | Both can enter β(t); still run VIF as robustness check |

> **Next — Step 3:** Cross-correlation / lead-lag analysis to determine whether articles or tone *lead* or *lag* Google Trends at weekly resolution.